# U05_P — Sequence Alignment — Exercises
**Course:** Concepts of Bioinformatics (CBBI-501) · **Unit 5:** Sequence Alignment and Database Searching
**Type:** Practice notebook · **Time:** 50 min · **Difficulty:** ★★☆
**Goes with:** Lectures L10–L15, Practicals S6–S7
**Before this:** U05_C1, U05_C2, U05_C3

In [ ]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)

NOTEBOOK_ID = "U05_P"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs" / NOTEBOOK_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

import Bio
print("Python", sys.version.split()[0], "| Biopython", Bio.__version__)

### Ex 5.1 ★☆☆ — Confirm Needleman-Wunsch initialization row and column
**Biology:** In global alignment, the margins represent end-gaps from aligning against leading insertions.  
**Task:** Given gap penalty $d = -3$, calculate the value of cell $F(4, 0)$ in the initialization column. Store in `col_val`.  
**Expected:** `col_val` as integer equal to $-12$.

In [ ]:
# TODO: replace the ... with your code
gap_penalty = -3
col_val = ...

In [ ]:
# Check — do not edit
assert col_val == -12, f"Expected -12, got {col_val}"
print("Ex 5.1 passed")

### Ex 5.2 ★☆☆ — Inspect BLOSUM62 score for sickle-cell substitution
**Biology:** The sickle-cell mutation replaces glutamic acid (E) with valine (V) at position 6.  
**Task:** Load `BLOSUM62` from `Bio.Align.substitution_matrices`. Extract the score for pair `('E', 'V')` into `e_v_score`, and self-score `('E', 'E')` into `e_e_score`.  
**Expected:** `e_v_score` is negative, `e_e_score` is positive.

In [ ]:
# TODO: replace the ... with your code
from Bio.Align import substitution_matrices

blosum62 = substitution_matrices.load("BLOSUM62")
e_v_score = ...
e_e_score = ...

In [ ]:
# Check — do not edit
assert e_v_score == -2, f"Expected -2, got {e_v_score}"
assert e_e_score == 5, f"Expected 5, got {e_e_score}"
print("Ex 5.2 passed")

### Ex 5.3 ★★☆ — Does the sickle-cell mutation change the pairwise alignment score?
**Biology:** HbS differs from HbA at residue 6 (E6V).  
**Task:** Align the first 20 residues of HbA (`hba_pep`) and HbS (`hbs_pep`) using `PairwiseAligner` with `BLOSUM62`, `open_gap_score=-10`, `extend_gap_score=-1`. Store the score in `score_diff` as `aligner.score(hba_pep, hba_pep) - aligner.score(hba_pep, hbs_pep)`.  
**Expected:** `score_diff` as a positive float.

In [ ]:
# TODO: replace the ... with your code
from Bio.Align import PairwiseAligner

hba_pep = "VHLTPEEKSAVTALWGKVNV"
hbs_pep = "VHLTPVEKSAVTALWGKVNV"

aligner = PairwiseAligner()
aligner.substitution_matrix = blosum62
aligner.open_gap_score = -10
aligner.extend_gap_score = -1

self_score = ...
variant_score = ...
score_diff = ...

In [ ]:
# Check — do not edit
assert score_diff > 0, "Self score must be strictly greater than mismatch score"
assert score_diff == 7.0, f"Expected score difference of 7.0 (5 - (-2)), got {score_diff}"
print("Ex 5.3 passed")

### Ex 5.4 ★★☆ — Local vs Global alignment of a truncated peptide against full HBB
**Biology:** A 15-aa peptide fragment aligned globally against a 147-aa protein is heavily penalized by end-gaps.  
**Task:** Align `fragment = 'TALWGKVNVDEVGGE'` against full `hbb_prot`. Compute global alignment score in `glob_score` and local alignment score in `loc_score`.  
**Expected:** `loc_score` is significantly higher than `glob_score`.

In [ ]:
# TODO: replace the ... with your code
from Bio import SeqIO

hbb_prot = str(SeqIO.read(DATA_DIR / "sequences" / "hbb_human_protein.fasta", "fasta").seq)
fragment = "TALWGKVNVDEVGGE"

aligner.mode = "global"
glob_score = ...

aligner.mode = "local"
loc_score = ...

In [ ]:
# Check — do not edit
assert loc_score > glob_score, "Local alignment score must exceed global score for a short fragment"
assert loc_score > 60.0, f"Expected local score > 60, got {loc_score}"
print("Ex 5.4 passed")

### Ex 5.5 ★★☆ — Filter BLAST hits by E-value and identity
**Biology:** Distinguishing paralogues (e.g. alpha-globin vs beta-globin) from close orthologues.  
**Task:** Load `data/expected/blast_hbb_vs_refseq.tsv` with pandas. Filter hits having `identity >= 70.0` and `evalue < 1e-50`. Store the count of filtered hits in `num_close_hits`.  
**Expected:** `num_close_hits` as integer >= 2.

In [ ]:
# TODO: replace the ... with your code
import pandas as pd

blast_df = pd.read_csv(DATA_DIR / "expected" / "blast_hbb_vs_refseq.tsv", sep="\t")
filtered_df = ...
num_close_hits = len(filtered_df)

In [ ]:
# Check — do not edit
assert num_close_hits >= 2, f"Expected at least 2 close globin hits, got {num_close_hits}"
print("Ex 5.5 passed")

### Ex 5.6 ★★★ — Calculate E-value for an observed bit score
**Biology:** Understanding how database expansion changes alignment statistical confidence.  
**Task:** Write a function `compute_evalue(m, n, bit_score)` using the formula $E = m \cdot n \cdot 2^{-S'}$. Calculate $E$ for $m=147$, $n=5 \times 10^7$, and $S'=80$. Store in `my_evalue`.  
**Expected:** `my_evalue` as a very small float ($< 10^{-13}$).

In [ ]:
# TODO: replace the ... with your code
def compute_evalue(m, n, bit_score):
    ...

my_evalue = compute_evalue(147, 50000000, 80)

In [ ]:
# Check — do not edit
assert isinstance(my_evalue, float)
assert my_evalue < 1e-13, f"Expected < 1e-13, got {my_evalue}"
print("Ex 5.6 passed")

## 5. Reflection Questions
1. Why does the Smith–Waterman recurrence include a zero floor ($H(i, j) = \max(0, ...)$), and what biological problem does this solve?
2. If the size of GenBank doubles over the next five years, what happens to the Bit Score and the E-value of an identical BLAST hit?